# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [ ]:
NAMES = "group11-section1"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

### Cost Function
Binary classification uses binary cross-entropy (log loss), which penalizes differences between predicted probabilities and binary labels. Multi-class classification uses categorical cross-entropy, which measures divergence between predicted probability distributions across classes and one-hot encoded targets. Regression uses mean squared error (MSE), which penalizes the squared continuous distance between predicted values and real-valued targets.

### Output Layer Changes
Binary classification requires 1 output unit with a sigmoid activation function to output probabilities bounded between 0 and 1. Multi-class classification requires as many output units as there are classes (K units) with a softmax activation function to produce a valid probability distribution that sums to 1. Regression requires 1 output unit with no activation function (identity/linear) so the network can predict unconstrained real values.

| Task | Output units | Output activation | Prediction |
|---|---|---|---|
| Binary | 1 | sigmoid (probability of class 1) | $p \ge 0.5 \Rightarrow 1$ |
| Multi-class | $K$ = number of classes (3 here) | softmax (probabilities summing to 1) | argmax over the $K$ units |
| Regression | 1 | none (identity / linear) so any real value can be output | the raw output |

### Data Standardization
Standardizing features to zero mean and unit variance ensures that features with larger original scales do not disproportionately dominate gradient updates. This stabilizes training, prevents vanishing or exploding gradients, and leads to faster, more uniform parameter convergence.

### Overfitting Detection
Overfitting is detected by monitoring both training and holdout metrics across epochs. If training loss continues to decrease and training accuracy improves while holdout loss starts increasing or holdout performance degrades, the model has begun memorizing training noise rather than generalizing.

In [36]:
# imports
from sklearn.datasets import make_moons
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd
import torch
from torch import relu
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
_X, _y = make_moons(n_samples=1000, noise=0.2, random_state=1234)
_X_train, _X_test, _y_train, _y_test = train_test_split(
    _X, _y, test_size=0.2, random_state=42
)

_scaler = StandardScaler()
_X_train_scaled = _scaler.fit_transform(_X_train)
_X_test_scaled = _scaler.transform(_X_test)

_X_train_t = torch.tensor(_X_train_scaled, dtype=torch.float32)
_y_train_t = torch.tensor(_y_train, dtype=torch.float32).unsqueeze(1)
_X_test_t = torch.tensor(_X_test_scaled, dtype=torch.float32)
_y_test_t = torch.tensor(_y_test, dtype=torch.float32).unsqueeze(1)

In [38]:
class BinaryMLP:
    def __init__(self, input_dim=2, hidden_dim=16, num_layers=1, activation="relu"):
        self.num_layers = num_layers
        self.activation = torch.relu if activation == "relu" else torch.tanh
        scale1 = (2.0 / input_dim) ** 0.5
        scale2 = (2.0 / hidden_dim) ** 0.5
        self.w1 = (torch.randn(input_dim, hidden_dim) * scale1).requires_grad_()
        self.b1 = torch.zeros(hidden_dim, requires_grad=True)
        if num_layers == 1:
            self.w2 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b2 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2]
        else:
            self.w2 = (torch.randn(hidden_dim, hidden_dim) * scale2).requires_grad_()
            self.b2 = torch.zeros(hidden_dim, requires_grad=True)
            self.w3 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b3 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2, self.w3, self.b3]

    def forward(self, X):
        h1 = self.activation(X @ self.w1 + self.b1)
        if self.num_layers == 1:
            return torch.sigmoid(h1 @ self.w2 + self.b2)
        h2 = self.activation(h1 @ self.w2 + self.b2)
        return torch.sigmoid(h2 @ self.w3 + self.b3)

    def cost(self, y_pred, y):
        eps = 1e-7
        y_pred = torch.clamp(y_pred, eps, 1.0 - eps)
        return -torch.mean(y * torch.log(y_pred) + (1.0 - y) * torch.log(1.0 - y_pred))

    def fit(self, X, y, epochs=200, lr=0.05, batch_size=32):
        n = X.shape[0]
        for epoch in range(epochs):
            indices = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = indices[i:i + batch_size]
                xb, yb = X[idx], y[idx]
                y_pred = self.forward(xb)
                loss = self.cost(y_pred, yb)
                loss.backward()
                with torch.no_grad():
                    for p in self.params:
                        p -= lr * p.grad
                        p.grad.zero_()

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).float()

In [39]:
_configs = [
    (1, 4, "relu"),
    (1, 8, "relu"),
    (1, 16, "relu"),
    (1, 32, "relu"),
    (2, 4, "relu"),
    (2, 8, "relu"),
    (2, 16, "relu"),
    (2, 32, "relu"),
    (1, 4, "tanh"),
    (1, 8, "tanh"),
    (1, 16, "tanh"),
    (1, 32, "tanh"),
    (2, 4, "tanh"),
    (2, 8, "tanh"),
    (2, 16, "tanh"),
    (2, 32, "tanh"),
]

_results = []
for _layers, _units, _act in _configs:
    torch.manual_seed(42)
    _model = BinaryMLP(input_dim=2, hidden_dim=_units, num_layers=_layers, activation=_act)
    _model.fit(_X_train_t, _y_train_t, epochs=200, lr=0.05, batch_size=32)
    _train_acc = (_model.predict(_X_train_t) == _y_train_t).float().mean().item()
    _test_acc = (_model.predict(_X_test_t) == _y_test_t).float().mean().item()
    _results.append({
        "hidden_layers": _layers,
        "hidden_units": _units,
        "activation": _act,
        "train_accuracy": round(_train_acc, 4),
        "holdout_accuracy": round(_test_acc, 4),
    })

binary_results_df = pd.DataFrame(_results)
print(binary_results_df)

    hidden_layers  hidden_units activation  train_accuracy  holdout_accuracy
0               1             4       relu          0.8700             0.860
1               1             8       relu          0.9700             0.980
2               1            16       relu          0.9712             0.980
3               1            32       relu          0.9700             0.985
4               2             4       relu          0.8675             0.860
5               2             8       relu          0.9737             0.980
6               2            16       relu          0.9737             0.980
7               2            32       relu          0.9775             0.990
8               1             4       tanh          0.9300             0.935
9               1             8       tanh          0.9650             0.980
10              1            16       tanh          0.9650             0.980
11              1            32       tanh          0.9700             0.990

Increasing the number of hidden units from 4 to 8 yields a sharp improvement in performance, moving accuracy from around 86 percent to 98 percent because 8 units provide enough capacity to separate the interlocking moon shapes. Expanding beyond 8 units to 16 and 32 units yields minimal additional benefit, as test accuracy plateaus around 98 to 99 percent.

Using two hidden layers instead of one provides only a marginal improvement once the network has at least 8 units, raising accuracy slightly from 98 percent to 99 percent. For smaller models with only 4 units, adding a second layer helps the network compute more complex compositional boundaries, particularly when using Tanh.

Both ReLU and Tanh achieve comparable top performance around 98 to 99 percent when given 8 or more units. At low capacity with only 4 units, Tanh outperforms ReLU (93.5 percent versus 86.0 percent) because its smooth, zero-centered sigmoid curve provides cleaner gradient flow and smoothly fits the curved boundary with fewer neurons.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$-\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [40]:
X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [41]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=1234, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

# one-hot encode the labels

# (OneHotEncoder needs 2D input, so I need to reshape to a column)
encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_holdout_oh = encoder.transform(y_holdout.reshape(-1, 1))

# making them tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train_oh = torch.tensor(y_train_oh, dtype=torch.float32)
y_holdout_oh = torch.tensor(y_holdout_oh, dtype=torch.float32)


class MultiClassMLP(object):

    def __init__(self, input_dim, hidden_dim, output_dim, n_hidden_layers=2, learning_rate=0.01, activation="relu"):
            
        # first hidden layer (always)
        self.W_1 = (0.1 * torch.randn(input_dim, hidden_dim)).requires_grad_()
                # multiply by 0.1 to scale the initial weights
        self.b_1 = torch.randn(hidden_dim, requires_grad=True)

        # second hidden layer (only used when n_hidden_layers == 2)
        if n_hidden_layers == 2:
            self.W_2 = (0.1 * torch.randn(hidden_dim, hidden_dim)).requires_grad_()
            self.b_2 = torch.randn(hidden_dim, requires_grad=True)

        # output layer (always)
        self.W_3 = (0.1 * torch.randn(hidden_dim, output_dim)).requires_grad_()
        self.b_3 = torch.randn(output_dim, requires_grad=True)

        self.n_hidden_layers = n_hidden_layers
        self.learning_rate = learning_rate
        self.activation = activation

    def forward(self, X):

        h_1 = torch.matmul(X, self.W_1) + self.b_1

        if self.activation == "relu":
            h_1 = relu(h_1)
        elif self.activation == "tanh":
            h_1 = torch.tanh(h_1)
        else:
            raise ValueError("activation must be 'relu' or 'tanh'")

        h = h_1  # output of the last hidden layer

        if self.n_hidden_layers == 2:
            h_2 = torch.matmul(h_1, self.W_2) + self.b_2

            if self.activation == "relu":
                h_2 = relu(h_2)
            elif self.activation == "tanh":
                h_2 = torch.tanh(h_2)

            h = h_2

        logits = torch.matmul(h, self.W_3) + self.b_3 # logits are the raw scores before softmax

        # softmax means each row becomes probabilities between 0 and 1 that sum to 1
        out = F.softmax(logits, dim=1) # dim = 1 reads across columns, i.e., each row sums to 1

        return out

    def cost(self, out, y):
        # categorical cross-entropy by hand: out = softmax probabilities, y = one-hot labels
        # clamp avoids log(0)
        return -torch.mean(torch.sum(y * torch.log(out.clamp(min=1e-15)), dim=1)) # categoricalcross-entropy loss

    def fit(self, X, y, n_epochs=150):

        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0]) # randperm generates a random permutation of indices for stochastic gradient descent

            for ii in randperm:
                x_batch = X[ii].unsqueeze(0)
                y_batch = y[ii].unsqueeze(0)   # y is one-hot, shape (1, 3)

                out = self.forward(x_batch)

                cost = self.cost(out, y_batch)

                cost.backward()

                with torch.no_grad():
                    self.W_1 -= self.learning_rate * self.W_1.grad
                    self.b_1 -= self.learning_rate * self.b_1.grad

                    if self.n_hidden_layers == 2:
                        self.W_2 -= self.learning_rate * self.W_2.grad
                        self.b_2 -= self.learning_rate * self.b_2.grad

                    self.W_3 -= self.learning_rate * self.W_3.grad
                    self.b_3 -= self.learning_rate * self.b_3.grad

                    # zero gradients
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()

                    if self.n_hidden_layers == 2:
                        self.W_2.grad.zero_()
                        self.b_2.grad.zero_()

                    self.W_3.grad.zero_()
                    self.b_3.grad.zero_()

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")

        return self

    def predict(self, X):

        with torch.no_grad(): # disable gradient calculation for prediction
            out = self.forward(X)
            predicted = torch.argmax(out, dim=1) # argmax b/c we want the class with the highest probability

        return predicted

In [42]:
results = []

configs = [
    (1, 4, "relu"),
    (1, 8, "relu"),
    (1, 16, "relu"),
    (1, 32, "relu"),
    (2, 4, "relu"),
    (2, 8, "relu"),
    (2, 16, "relu"),
    (2, 32, "relu"),
    (1, 4, "tanh"),
    (1, 8, "tanh"),
    (1, 16, "tanh"),
    (1, 32, "tanh"),
    (2, 4, "tanh"),
    (2, 8, "tanh"),
    (2, 16, "tanh"),
    (2, 32, "tanh"),
]

for n_hidden_layers, hidden_dim, activation in configs:
    torch.manual_seed(42)

    model = MultiClassMLP(
        input_dim=X_train.shape[1],
        hidden_dim=hidden_dim,
        output_dim=3,
        n_hidden_layers=n_hidden_layers,
        learning_rate=0.01,
        activation=activation
    )

    model.fit(X_train, y_train_oh, n_epochs=200)

    with torch.no_grad():
        train_cost = model.cost(
            model.forward(X_train), y_train_oh
        ).item()

        holdout_cost = model.cost(
            model.forward(X_holdout), y_holdout_oh
        ).item()

    results.append({
        "Hidden layers": n_hidden_layers,
        "Neurons per layer": hidden_dim,
        "Activation": activation,
        "Train Accuracy": round(
            accuracy_score(y_train, model.predict(X_train)), 4
        ),
        "Holdout Accuracy": round(
            accuracy_score(y_holdout, model.predict(X_holdout)), 4
        ),
        "Train cost": train_cost,
        "Holdout cost": holdout_cost
    })

Epoch 0, Cost: 0.5441
Epoch 10, Cost: 0.1113
Epoch 20, Cost: 0.2320
Epoch 30, Cost: 0.1300
Epoch 40, Cost: 0.1723
Epoch 50, Cost: 2.3785
Epoch 60, Cost: 0.0944
Epoch 70, Cost: 0.1837
Epoch 80, Cost: 0.2260
Epoch 90, Cost: 2.0551
Epoch 100, Cost: 0.8748
Epoch 110, Cost: 0.3875
Epoch 120, Cost: 0.2192
Epoch 130, Cost: 1.5664
Epoch 140, Cost: 0.0556
Epoch 150, Cost: 0.4724
Epoch 160, Cost: 0.2676
Epoch 170, Cost: 0.0914
Epoch 180, Cost: 0.1035
Epoch 190, Cost: 1.9446
Epoch 0, Cost: 0.3784
Epoch 10, Cost: 0.1457
Epoch 20, Cost: 0.1166
Epoch 30, Cost: 0.1760
Epoch 40, Cost: 0.0930
Epoch 50, Cost: 0.2454
Epoch 60, Cost: 0.2837
Epoch 70, Cost: 0.1097
Epoch 80, Cost: 0.3256
Epoch 90, Cost: 0.3000
Epoch 100, Cost: 0.1093
Epoch 110, Cost: 0.1538
Epoch 120, Cost: 0.4100
Epoch 130, Cost: 1.1168
Epoch 140, Cost: 0.2327
Epoch 150, Cost: 1.0328
Epoch 160, Cost: 0.1419
Epoch 170, Cost: 0.0864
Epoch 180, Cost: 2.3807
Epoch 190, Cost: 3.1625
Epoch 0, Cost: 0.9346
Epoch 10, Cost: 0.5134
Epoch 20, Cost: 0

In [43]:
results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))

 Hidden layers  Neurons per layer Activation  Train Accuracy  Holdout Accuracy  Train cost  Holdout cost
             1                  4       relu          0.8113             0.800      0.5755        0.6088
             1                  8       relu          0.8175             0.815      0.5644        0.6058
             1                 16       relu          0.8013             0.800      0.5778        0.6160
             1                 32       relu          0.8225             0.815      0.5374        0.5950
             2                  4       relu          0.8075             0.795      0.5536        0.5992
             2                  8       relu          0.8275             0.800      0.5530        0.6572
             2                 16       relu          0.8275             0.805      0.5290        0.6229
             2                 32       relu          0.8275             0.805      0.5352        0.6093
             1                  4       tanh          0

### Results and Discussion

**Hidden units.** More units steadily lowered training cost (one-layer ReLU: 0.579 → 0.564 → 0.559 → 0.554 for 4 → 8 → 16 → 32 units; one-layer tanh: 0.559 → 0.555 → 0.552 → 0.549), but holdout cost did not follow. For one-layer ReLU holdout cost stayed flat at 0.615–0.617, and for two-layer tanh it rose with width (0.579 → 0.588 → 0.610 → 0.612) while train cost fell, which is the classic sign of the wider network starting to fit noise. Accuracy tells supports this claim because train accuracy moves only between 0.809 and 0.821, and holdout accuracy stays between 0.790 and 0.805 for all sixteen models (a range of just three holdout samples). Four to eight units are already enough for this 2-feature problem.

`flip_y=0.15` randomly reassigns 15% of the labels, and the three clusters overlap, so no model can get much above ~0.85. Every model is already close to this ceiling, so the differences between configurations are small and mostly noise.

**One vs. two layers.** A second layer lowered training cost in 7 of the 8 matched pairs (mean 0.559 → 0.545), but helped the holdout set only for small widths: ReLU 4 (0.617 → 0.607), tanh 4 (0.614 → 0.579) and tanh 8 (0.604 → 0.588). At 16 and 32 units the two-layer model had a *higher* holdout cost in all four pairs (e.g. ReLU 32: 0.615 → 0.628), and its train–holdout gap widened (ReLU 32: 0.061 → 0.092). So depth adds capacity, which helps a narrow network but leads a wide one to overfit. Mean holdout accuracy is essentially unchanged (0.796 vs. 0.798).

**ReLU vs. tanh.** The two activations reach the same accuracy (mean train 0.815 ReLU vs. 0.818 tanh; mean holdout 0.797 vs. 0.796), but tanh has a lower holdout cost (0.602 vs. 0.617), and the single best holdout cost is the two-layer tanh network with 4 units (0.579). Tanh's smooth, bounded output produces softer, better-calibrated probabilities, while ReLU's unbounded, piecewise-linear units can push some logits to extreme values, producing over-confident wrong predictions that cross-entropy penalizes heavily.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [44]:
_X, _y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

_X_train, _X_test, _y_train, _y_test = train_test_split(
    _X, _y, test_size=0.2, random_state=42
)

_scaler = StandardScaler()
_X_train_scaled = _scaler.fit_transform(_X_train)
_X_test_scaled = _scaler.transform(_X_test)

_X_train_t = torch.tensor(_X_train_scaled, dtype=torch.float32)
_y_train_t = torch.tensor(_y_train, dtype=torch.float32).unsqueeze(1)
_X_test_t = torch.tensor(_X_test_scaled, dtype=torch.float32)
_y_test_t = torch.tensor(_y_test, dtype=torch.float32).unsqueeze(1)

In [45]:
class RegressionMLP:
    def __init__(self, input_dim=2, hidden_dim=16, num_layers=1, activation="relu"):
        self.num_layers = num_layers
        self.activation = torch.relu if activation == "relu" else torch.tanh
        
        scale1 = (2.0 / input_dim) ** 0.5
        scale2 = (2.0 / hidden_dim) ** 0.5
        
        self.w1 = (torch.randn(input_dim, hidden_dim) * scale1).requires_grad_()
        self.b1 = torch.zeros(hidden_dim, requires_grad=True)
        
        if num_layers == 1:
            self.w2 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b2 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2]
        else:
            self.w2 = (torch.randn(hidden_dim, hidden_dim) * scale2).requires_grad_()
            self.b2 = torch.zeros(hidden_dim, requires_grad=True)
            self.w3 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b3 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2, self.w3, self.b3]

    def forward(self, X):
        h1 = self.activation(X @ self.w1 + self.b1)
        if self.num_layers == 1:
            return h1 @ self.w2 + self.b2  # No activation on output for regression
        h2 = self.activation(h1 @ self.w2 + self.b2)
        return h2 @ self.w3 + self.b3

    def cost(self, y_pred, y):
        return torch.mean((y_pred - y) ** 2)  # Mean Squared Error (MSE)

    def fit(self, X, y, epochs=150, lr=0.001, batch_size=32):
        n = X.shape[0]
        for epoch in range(epochs):
            indices = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = indices[i:i + batch_size]
                xb, yb = X[idx], y[idx]
                
                y_pred = self.forward(xb)
                loss = self.cost(y_pred, yb)
                loss.backward()
                
                with torch.no_grad():
                    for p in self.params:
                        if p.grad is not None:
                            torch.nn.utils.clip_grad_norm_([p], max_norm=1.0)
                            p -= lr * p.grad
                            p.grad.zero_()
    def predict(self, X):
        with torch.no_grad():
            return self.forward(X)

In [46]:
_configs = [
    (1, 4, "relu"), (1, 8, "relu"), (1, 16, "relu"), (1, 32, "relu"),
    (2, 4, "relu"), (2, 8, "relu"), (2, 16, "relu"), (2, 32, "relu"),
    (1, 4, "tanh"), (1, 8, "tanh"), (1, 16, "tanh"), (1, 32, "tanh"),
    (2, 4, "tanh"), (2, 8, "tanh"), (2, 16, "tanh"), (2, 32, "tanh"),
]

_results = []
for _layers, _units, _act in _configs:
    torch.manual_seed(42)
    _model = RegressionMLP(input_dim=2, hidden_dim=_units, num_layers=_layers, activation=_act)
    _model.fit(_X_train_t, _y_train_t, epochs=150, lr=0.001, batch_size=32)
    
    train_preds = _model.predict(_X_train_t)
    test_preds = _model.predict(_X_test_t)
    
    train_mse = mean_squared_error(_y_train_t.numpy(), train_preds.numpy())
    test_mse = mean_squared_error(_y_test_t.numpy(), test_preds.numpy())
    
    _results.append({
        "hidden_layers": _layers,
        "hidden_units": _units,
        "activation": _act,
        "train_mse": round(train_mse, 4),
        "holdout_mse": round(test_mse, 4),
    })

regression_results_df = pd.DataFrame(_results)
print(regression_results_df)

    hidden_layers  hidden_units activation   train_mse  holdout_mse
0               1             4       relu   9134.1221    8315.3926
1               1             8       relu   8921.8350    7985.6938
2               1            16       relu   9133.9092    8202.8896
3               1            32       relu   8637.9346    7756.3164
4               2             4       relu   7606.2402    7246.2476
5               2             8       relu   6015.0581    5893.6309
6               2            16       relu   6768.2764    6409.0439
7               2            32       relu   5928.1465    5770.8354
8               1             4       tanh  10373.5186    9175.3340
9               1             8       tanh  10070.3652    8927.2959
10              1            16       tanh  10083.6309    8933.8271
11              1            32       tanh   9461.0713    8428.2598
12              2             4       tanh  10122.4033    8981.8359
13              2             8       tanh   988

1. How does the number of hidden units affect performance? As the number of hidden units increases, the MSE generally decreases for both single layer and 2 layer configurations. Giving the model more hidden units increases its capacity to learn more complex features from the data, resulting in a tighter fit and lower error.
2. What changes when using two layers instead of one? Using two layers results in an improvement in performance with lower MSE values. The extra depth allows the network to more effectively capture non-linear relationships in the data.
3. How does the activation function (ReLU vs. Tanh) influence results? The ReLU activation function achieved lower MSE values than Tanh. This is due to Tanh activations possibly suffering from saturation in deeper manually-updated networks, whereas ReLU allows gradients to flow more freely leading to more effective optimization.

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

A common base class called BaseMLP can encapsulate the shared logic of parameter initialization, hidden layer transformations, and the stochastic gradient descent training loop. The base class manages the weight tensors, bias vectors, gradient zeroing, and backward step execution across mini-batches since this procedure is identical regardless of the task.

Task-specific classes such as BinaryMLP, MultiClassMLP, and RegressionMLP can inherit from BaseMLP and override only the output transformation and the loss function. BinaryMLP defines binary cross-entropy loss with sigmoid thresholding, MultiClassMLP defines categorical cross-entropy loss with softmax argmax class prediction, and RegressionMLP defines mean squared error loss with direct linear pass-through. This design eliminates duplicated training and layer-propagation code across all three models while isolating task-specific behavior to a few lines per subclass.

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

Our group divided the lab into separate parts so that each member could focus on one section, which improved the quality of everyone’s individual work because we were able to explore our parts in greater depth. The main inefficiency we encountered was communication and scheduling, as we did not meet as a group until the Monday before the lab was due. This left us with limited time to compare results, resolve inconsistencies between sections, and complete the write-up together, however, we do not believe this hindered results. In the next lab, we will schedule a group meeting within the first few days of the lab being released, ideally during the same week, and we will set a shared calendar reminder and a mid-week check-in in our group chat so that everyone’s progress is visible well before the deadline.